In [89]:
import pandas as pd
import re
import nltk  # ← TAMBAHIN INI
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

# Initialize
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Download NLTK resources (jalankan sekali aja)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Load stopwords
stop_words = set(stopwords.words('indonesian'))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\WIN10\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\WIN10\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\WIN10\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [90]:
# Load raw data
df = pd.read_csv("ALL_DATA_BERITA_22-24.csv")

print(f"Total data awal: {len(df)}")
print(f"\nKolom: {df.columns.tolist()}")
print(f"\nSample data:")
df.head()

Total data awal: 17654

Kolom: ['tanggal', 'kategori', 'judul', 'url', 'konten']

Sample data:


,tanggal,kategori,judul,url,konten
0,2022-01-01,ekonomi,SKK Migas Targetkan Pengeboran 700 Sumur di Ta...,https://finance.detik.com/energi/d-5880032/skk...,Dalam rangka mengoptimalkan lifting migas di t...
1,2022-01-01,ekonomi,"Ekspor Batu Bara Dilarang Sementara, Pengusaha...",https://finance.detik.com/energi/d-5880129/eks...,Pemerintah menyetop sementara ekspor batu bara...
2,2022-01-01,ekonomi,Jaksa Agung Puji Erick Thohir Bantu Bongkar Ko...,https://finance.detik.com/moneter/d-5880248/ja...,Jaksa Agung Burhanuddin memberikan pujian kepa...
3,2022-01-01,ekonomi,"Terbang dengan Ancaman Omicron, Pilot Maskapai...",https://finance.detik.com/berita-ekonomi-bisni...,Maskapai penerbangan Amerika Serikat (AS) Unit...
4,2022-01-01,ekonomi,"450 Karyawannya Mogok Kerja, Warren Buffett Ma...",https://finance.detik.com/berita-ekonomi-bisni...,Ratusan karyawan dari salah satu anak perusaha...


# Basic Cleaning

In [91]:
# Remove duplicates
df = df.drop_duplicates(subset=['judul', 'konten'], keep='first')
print(f"Setelah remove duplicates: {len(df)}")

# Remove null values
df = df.dropna(subset=['judul', 'konten'])
print(f"Setelah remove null: {len(df)}")

# Convert tanggal to datetime
df['tanggal'] = pd.to_datetime(df['tanggal'])

# Reset index
df = df.reset_index(drop=True)

print(f"\nData setelah basic cleaning: {len(df)} rows")

Setelah remove duplicates: 17411
Setelah remove null: 17114

Data setelah basic cleaning: 17114 rows


# Filter Berita Relevan dengan Saham

**Justifikasi Akademis:**
> Penelitian ini menggunakan keyword domain-specific yang diadaptasi dari literatur analisis pasar modal Indonesia (Loughran & McDonald, 2011; Koto & Rahmaningtyas, 2017) untuk memfilter berita yang relevan dengan IHSG. Keyword mencakup istilah core stock market (saham, IHSG, BEI), instrumen finansial (dividen, obligasi), market actions (IPO, listing), dan indikator ekonomi makro yang mempengaruhi pergerakan saham (inflasi, suku bunga, kurs).

In [ ]:
saham_keywords= [
    "saham",
    "pasar",
    "bursa",
    "indeks",
    "ihsg",
    "ekuitas",
    "dividen",
    "emiten",
    "korporasi",
    "manajemen",
    "pemegang saham",
    "laba",
    "arus kas",
    "aset",
    "liabilitas",
    "risiko",
    "volatilitas",
    "fluktuasi",
    "eksposur",
    "suku bunga",
    "inflasi",
    "moneter",
    "regulasi",
    "stimulus",
    "aksi korporasi",
    "analisis fundamental",
    "analisis teknikal",
    "annual report",
    "arbitrase",
    "ask price",
    "bid price",
    "blue chip",
    "bon",
    "breakout",
    "dividen",
    "capital gain",
    "capital loss",
    "foreign investor",
    "fundamental value",
    "ipo",
    "indeks saham",
    "inflasi",
    "liquidity",
    "lot",
    "market order",
    "limit order",
    "margin trading",
    "moving average",
    "mutual fund",
    "reksa dana",
    "resistance",
    "support",
    "right issue",
    "return on equity",
    "risk",
    "stock split",
    "short selling",
    "stop loss",
    "ticker symbol",
    "trading volume",
    "trend",
    "turnover",
    "undervalued",
    "overvalued",
    "volatilitas",
    "watchlist",
    "yield",
    "beta",
    "book value",
    "cash flow",
    "day trading",
    "equity",
    "earnings per share",
    "eps",
    "ex-dividend date",
    "falling knife",
    "saham",
    "OJK"
]

core_keywords = [
    "saham", "harga saham",
    "bursa efek",
    "pasar modal",
    "indeks", 
    "ihsg", 
    "indeks harga saham gabungan"
]



In [104]:
def contains_keyword(text, keywords):
    if pd.isna(text):
        return False
    text = str(text).lower()
    return any(k.lower() in text for k in keywords)

In [106]:
df['is_stock_related'] = df.apply(
    lambda row: (
        (
            contains_keyword(row['judul'], core_keywords) or
            contains_keyword(row['konten'], core_keywords)
        )
        and
        (
            contains_keyword(row['judul'], saham_keywords) or
            contains_keyword(row['konten'], saham_keywords)
        )
    ),
    axis=1
)

In [107]:
total_berita = len(df)
total_relevan = df['is_stock_related'].sum()
total_tidak_relevan = total_berita - total_relevan

print(f"Total berita           : {total_berita}")
print(f"Berita relevan saham   : {total_relevan} ({total_relevan/total_berita*100:.2f}%)")
print(f"Berita tidak relevan   : {total_tidak_relevan} ({total_tidak_relevan/total_berita*100:.2f}%)")


Total berita           : 17114
Berita relevan saham   : 1511 (8.83%)
Berita tidak relevan   : 15603 (91.17%)


In [108]:
df_stock = df[df['is_stock_related'] == True].copy()
df_stock.reset_index(drop=True, inplace=True)

In [52]:
df_non_stock = df[df['is_stock_related'] == False].copy()
df_non_stock.reset_index(drop=True, inplace=True)

In [109]:
print("CONTOH BERITA RELEVAN SAHAM")
print("="*80)

for i in range(min(5, len(df_stock))):
    print(f"\n{i+1}. {df_stock.loc[i, 'judul']}")
    print(f"   Tanggal: {df_stock.loc[i, 'tanggal']}")
    print(f"   Preview: {df_stock.loc[i, 'konten'][:200]}...")
    print("-"*80)

CONTOH BERITA RELEVAN SAHAM

1. Jaksa Agung Puji Erick Thohir Bantu Bongkar Korupsi Jiwasraya dan Asabri
   Tanggal: 2022-01-01 00:00:00
   Preview: Jaksa Agung Burhanuddin memberikan pujian kepada Menteri Badan Usaha Milik Negara (BUMN) Erick Thohir karena telah berkontribusi membongkar skandal korupsi PT Asuransi Jiwasraya dan PT ASABRI (Persero...
--------------------------------------------------------------------------------

2. Aturan Harga hingga Penyaluran Premium Dirombak Jokowi
   Tanggal: 2022-01-02 00:00:00
   Preview: Presiden Joko Widodo (Jokowi) telah mengeluarkan Peraturan Presiden Nomor 117 Tahun 2021 tentang Perubahan Ketiga Atas Peraturan Presiden Nomor 191 Tahun 2OI4 tentang Penyediaan, Pendistribusian dan H...
--------------------------------------------------------------------------------

3. Presiden Joko Widodo (Jokowi) telah mengeluarkan Peraturan Presiden Nomor 117 Tahun 2021 tentang Perubahan Ketiga Atas Peraturan Presiden Nomor 191 Tahun 2OI4 tentang Penyedi

In [110]:
df_stock.to_csv("BERITA_SAHAM_KEYWORD_MIRAE.csv")

# Text Preprocessing

In [ ]:
def preprocess_text(text):
    """
    Text preprocessing untuk analisis sentimen berita finansial
    
    Steps:
    1. Tokenization: split menjadi token
    2. Stopword Removal: hapus kata umum
    3. Stemming: normalisasi ke bentuk dasar
    
    Note: Angka/numbers dipertahankan untuk preserve magnitude information
    """
    if pd.isna(text):
        return ""
    
    # Tokenization
    tokens = re.findall(r'\b[\w\.]+\b', text)
    
    # Stopword removal & Stemming
    result = []
    for token in tokens:
        if token in stop_words:
            continue
        
        # Keep numbers (for magnitude)
        if any(c.isdigit() for c in token):
            result.append(token)
        # Stem words
        elif token.isalpha() and len(token) > 2:
            result.append(stemmer.stem(token))
    
    return ' '.join(result)

In [57]:
print("\nPreprocessing text...")
df_stock['konten_clean'] = df_stock['konten'].apply(preprocess_text)

# Remove empty
df_stock = df_stock[df_stock['konten_clean'].str.len() > 0]

print(f"✓ Processed {len(df_stock)} articles")

# Show example
if len(df_stock) > 0:
    print("\nExample:")
    print("="*80)
    print(f"Original:\n{df_stock.iloc[0]['konten'][:200]}...")
    print(f"\nCleaned:\n{df_stock.iloc[0]['konten_clean'][:200]}...")
    print("="*80)

df_stock.to_csv("RELEVAN_BERITA_SAHAM_FIX_BANGET.csv")


Preprocessing text...
✓ Processed 1511 articles

Example:
Original:
Jaksa Agung Burhanuddin memberikan pujian kepada Menteri Badan Usaha Milik Negara (BUMN) Erick Thohir karena telah berkontribusi membongkar skandal korupsi PT Asuransi Jiwasraya dan PT ASABRI (Persero...

Cleaned:
jaksa agung burhanuddin puji menteri badan usaha milik negara bumn erick thohir kontribusi bongkar skandal korupsi asuransi jiwasraya asabri persero terima kasih menteri badan usaha milik negara bapak...


In [58]:
df = pd.read_csv("RELEVAN_BERITA_SAHAM_FIX_BANGET.csv")
df.head()

,Unnamed: 0,tanggal,kategori,judul,url,konten,is_stock_related,konten_clean
0,0,2022-01-01,ekonomi,Jaksa Agung Puji Erick Thohir Bantu Bongkar Ko...,https://finance.detik.com/moneter/d-5880248/ja...,Jaksa Agung Burhanuddin memberikan pujian kepa...,True,jaksa agung burhanuddin puji menteri badan usa...
1,1,2022-01-02,ekonomi,Aturan Harga hingga Penyaluran Premium Diromba...,https://finance.detik.com/energi/d-5881226/atu...,Presiden Joko Widodo (Jokowi) telah mengeluark...,True,presiden joko widodo jokowi keluar atur presid...
2,2,2022-01-02,ekonomi,Presiden Joko Widodo (Jokowi) telah mengeluark...,https://finance.detik.com/energi/d-5881226/atu...,Presiden Joko Widodo (Jokowi) telah mengeluark...,True,presiden joko widodo jokowi keluar atur presid...
3,3,2022-01-03,ekonomi,"Genjot Bisnis Data Center, Telkom Ambil Alih S...",https://finance.detik.com/bursa-dan-valas/d-58...,PT Telkom Indonesia (Persero) Tbk (TLKM) menga...,True,telkom indonesia persero tbk tlkm ambil alih s...
4,4,2022-01-04,ekonomi,Ini Pesan Penting Luhut buat OJK,https://finance.detik.com/moneter/d-5884263/in...,Menteri Koordinator Kemaritiman dan Investasi ...,True,menteri koordinator maritim investasi luhut bi...
